# Cyber Sachet Evaluation Workflow
Notebook-first workflow for: scenario generation, batch runs, manual labels, and LLM judge validation.

In [ ]:
from pathlib import Path
import csv
import json

scenarios_path = Path('evaluation/data/scenarios.csv')
results_path = Path('evaluation/data/results.jsonl')
labels_path = Path('evaluation/data/labels.csv')
print('scenarios exists:', scenarios_path.exists())
print('results exists:', results_path.exists())
print('labels exists:', labels_path.exists())

## 1) Ensure at least 50 scenarios

In [ ]:
with scenarios_path.open('r', encoding='utf-8', newline='') as f:
    rows = list(csv.DictReader(f))
print('scenario_count =', len(rows))
rows[:3]

## 2) Batch run all scenarios

In [ ]:
import asyncio
from evaluation.batch_run import run_batch

await run_batch(
    scenarios_path=Path('evaluation/data/scenarios.csv'),
    out_dir=Path('evaluation/data'),
    model='gpt-4o-mini',
    limit=None,
)

## 3) Launch labeling tool (new terminal command)
Run this in terminal:
`streamlit run evaluation/label_evals.py`
Label at least 15 rows with `good` / `bad` and failure category.

## 4) Validate judge and metrics

In [ ]:
from evaluation.scripts.validate_judge import main_async

await main_async(
    results_path=Path('evaluation/data/results.jsonl'),
    labels_path=Path('evaluation/data/labels.csv'),
    out_dir=Path('evaluation/reports'),
)

In [ ]:
metrics = json.loads(Path('evaluation/reports/judge_metrics.json').read_text(encoding='utf-8'))
report = json.loads(Path('evaluation/reports/judge_iteration_report.json').read_text(encoding='utf-8'))
print(json.dumps(metrics, indent=2))
print('fixed disagreement example:', report.get('fixed_disagreement_example'))